# 01 - Carga y normalización global

Este notebook transforma todos los JSON detectados en dos tablas principales:

- `pycefr_constructs_global.csv`: constructos detectados por PyCEFR.
- `radon_functions_global.csv`: funciones detectadas por Radon.

Este paso es la base de todo el análisis posterior.

In [1]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LEVEL_ORDER = {"A1": 1, "A2": 2, "B1": 3, "B2": 4, "C1": 5, "C2": 6}
LEVEL_ORDER_INV = {v: k for k, v in LEVEL_ORDER.items()}
RADON_RANK_ORDER = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6}
RADON_RANK_ORDER_INV = {v: k for k, v in RADON_RANK_ORDER.items()}

def clean_file_name(path):
    """Devuelve un nombre de fichero comparable entre herramientas."""
    return Path(str(path)).name

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def detect_json_kind(path):
    """Intenta detectar si un JSON parece de Radon o de PyCEFR."""
    try:
        data = load_json(path)
    except Exception:
        return "unknown"
    # Buscar un registro de ejemplo dentro del árbol project/file/[records]
    for project, files in data.items():
        if not isinstance(files, dict):
            continue
        for file_path, records in files.items():
            if isinstance(records, list) and records:
                rec = records[0]
                if isinstance(rec, dict):
                    if {"Class", "Start Line", "End Line", "Level"}.issubset(set(rec.keys())):
                        return "pycefr"
                    if {"type", "rank", "complexity", "lineno", "endline"}.issubset(set(rec.keys())):
                        return "radon"
    return "unknown"


def normalize_pycefr_json(data, source_file=None, case_name=None):
    rows = []
    for project, files in data.items():
        if not isinstance(files, dict):
            continue
        for file_path, records in files.items():
            if not isinstance(records, list):
                continue
            for rec in records:
                if not isinstance(rec, dict):
                    continue
                level = rec.get("Level")
                rows.append({
                    "case": case_name,
                    "project": project,
                    "file": str(file_path),
                    "file_name": clean_file_name(file_path),
                    "class": rec.get("Class"),
                    "start_line": pd.to_numeric(rec.get("Start Line"), errors="coerce"),
                    "end_line": pd.to_numeric(rec.get("End Line"), errors="coerce"),
                    "displacement": pd.to_numeric(rec.get("Displacement"), errors="coerce"),
                    "level": level,
                    "level_num": LEVEL_ORDER.get(level),
                    "source_file": str(source_file) if source_file else None,
                })
    return pd.DataFrame(rows)

def normalize_radon_json(data, source_file=None, case_name=None, include_closures=True):
    rows = []
    def add_rec(project, file_path, rec, parent=None, depth=0):
        if not isinstance(rec, dict):
            return
        rank = rec.get("rank")
        rows.append({
            "case": case_name,
            "project": project,
            "file": str(file_path),
            "file_name": clean_file_name(file_path),
            "type": rec.get("type"),
            "name": rec.get("name"),
            "complexity": pd.to_numeric(rec.get("complexity"), errors="coerce"),
            "rank": rank,
            "rank_num": RADON_RANK_ORDER.get(rank),
            "lineno": pd.to_numeric(rec.get("lineno"), errors="coerce"),
            "endline": pd.to_numeric(rec.get("endline"), errors="coerce"),
            "col_offset": pd.to_numeric(rec.get("col_offset"), errors="coerce"),
            "parent": parent,
            "closure_depth": depth,
            "source_file": str(source_file) if source_file else None,
        })
        if include_closures:
            for child in rec.get("closures", []) or []:
                add_rec(project, file_path, child, parent=rec.get("name"), depth=depth+1)
    for project, files in data.items():
        if not isinstance(files, dict):
            continue
        for file_path, records in files.items():
            if not isinstance(records, list):
                continue
            for rec in records:
                add_rec(project, file_path, rec)
    return pd.DataFrame(rows)

def find_case_folders(processed_dir=PROCESSED_DIR):
    if not processed_dir.exists():
        return []
    return sorted([p for p in processed_dir.iterdir() if p.is_dir()])

def load_all_processed():
    pycefr_parts = []
    radon_parts = []
    catalog_rows = []
    for case_dir in find_case_folders():
        for path in sorted(case_dir.rglob("*.json")):
            kind = detect_json_kind(path)
            catalog_rows.append({"case": case_dir.name, "path": str(path), "kind": kind})
            if kind == "pycefr":
                pycefr_parts.append(normalize_pycefr_json(load_json(path), path, case_dir.name))
            elif kind == "radon":
                radon_parts.append(normalize_radon_json(load_json(path), path, case_dir.name))
    pycefr = pd.concat(pycefr_parts, ignore_index=True) if pycefr_parts else pd.DataFrame()
    radon = pd.concat(radon_parts, ignore_index=True) if radon_parts else pd.DataFrame()
    catalog = pd.DataFrame(catalog_rows)
    return catalog, pycefr, radon

In [2]:
catalog, df_pycefr, df_radon = load_all_processed()

print("Catálogo:", catalog.shape)
print("PyCEFR:", df_pycefr.shape)
print("Radon:", df_radon.shape)

Catálogo: (2, 3)
PyCEFR: (1176, 11)
Radon: (109, 15)


In [3]:
display(catalog.head())
display(df_pycefr.head())
display(df_radon.head())


,case,path,kind
0,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...,pycefr
1,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...,radon


,case,project,file,file_name,class,start_line,end_line,displacement,level,level_num,source_file
0,python-beginner-programming-exercises,11-Create-A-New-Function,test.py,test.py,'range' call function,25,25,13,A2,2,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...
1,python-beginner-programming-exercises,11-Create-A-New-Function,test.py,test.py,Files --> 'open' call function,32,32,9,A2,2,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...
2,python-beginner-programming-exercises,11-Create-A-New-Function,test.py,test.py,Files --> 'read' call function,33,33,18,A2,2,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...
3,python-beginner-programming-exercises,11-Create-A-New-Function,test.py,test.py,Simple Atributte,11,11,1,A2,2,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...
4,python-beginner-programming-exercises,11-Create-A-New-Function,test.py,test.py,Simple Atributte,20,20,1,A2,2,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...


,case,project,file,file_name,type,name,complexity,rank,rank_num,lineno,endline,col_offset,parent,closure_depth,source_file
0,python-beginner-programming-exercises,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,test.py,function,test_for_return,4,A,1,21,26,0,None,0,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...
1,python-beginner-programming-exercises,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,test.py,function,test_function_exists,2,A,1,12,17,0,None,0,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...
2,python-beginner-programming-exercises,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,test.py,function,test_for_type_random,2,A,1,31,35,0,None,0,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...
3,python-beginner-programming-exercises,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,test.py,function,test_function_called_for,2,A,1,41,41,0,None,0,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...
4,python-beginner-programming-exercises,python-beginner-programming-exercises,/home/juan/Documents/Analisis-CC-PyCEFR/python...,solution.hide.py,function,generate_random,1,A,1,4,6,0,None,0,/home/juan/Documents/Analisis-CC-PyCEFR/data/p...


In [5]:
print("Casos con PyCEFR:", sorted(df_pycefr["case"].dropna().unique()) if not df_pycefr.empty else [])
print("Casos con Radon:", sorted(df_radon["case"].dropna().unique()) if not df_radon.empty else [])

Casos con PyCEFR: ['python-beginner-programming-exercises']
Casos con Radon: ['python-beginner-programming-exercises']


In [6]:
if not df_pycefr.empty:
    display(df_pycefr.isna().sum().sort_values(ascending=False))
if not df_radon.empty:
    display(df_radon.isna().sum().sort_values(ascending=False))

case            0
project         0
file            0
file_name       0
class           0
start_line      0
end_line        0
displacement    0
level           0
level_num       0
source_file     0
dtype: int64

parent           109
project            0
case               0
file_name          0
type               0
name               0
file               0
complexity         0
rank               0
lineno             0
rank_num           0
endline            0
col_offset         0
closure_depth      0
source_file        0
dtype: int64

In [7]:
df_pycefr.to_csv(OUTPUT_DIR / "pycefr_constructs_global.csv", index=False)
df_radon.to_csv(OUTPUT_DIR / "radon_functions_global.csv", index=False)
catalog.to_csv(OUTPUT_DIR / "catalog_global.csv", index=False)
print("Archivos guardados en", OUTPUT_DIR)

Archivos guardados en /home/juan/Documents/Analisis-CC-PyCEFR/outputs


## Comentario para la memoria

La normalización permite trabajar con formatos homogéneos aunque las herramientas generen estructuras distintas. PyCEFR se representa como una tabla de constructos sintácticos, mientras que Radon se representa como una tabla de funciones con su complejidad ciclomática. Esta diferencia de granularidad condiciona la comparación y se tendrá en cuenta en los capítulos de metodología y resultados.